# Lab 2: When a Label Is a Measurement

Sentiment labels can look like facts attached to sentences. DynaSent instead
records several people's judgments. Today we ask:

> **What changes when we treat a label as a measurement rather than truth?**

We use one dataset throughout. Each sentence has five judgments—`negative`,
`neutral`, or `positive`—plus a frozen model probability vector. The data load
directly from the course GitHub repository. No upload or local path is needed.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import label_binarize

DATA_URL = 'https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/agent/week02-colab-preview/data/course/dynasent/items.csv'
items = pd.read_csv(DATA_URL)

LABELS = ["negative", "neutral", "positive"]
VOTE_COLS = [f"{label}_votes" for label in LABELS]
MODEL_COLS = [f"model_p_{label}" for label in LABELS]

assert len(items) == 72
assert items[VOTE_COLS].sum(axis=1).eq(5).all()
assert np.allclose(items[MODEL_COLS].sum(axis=1), 1)

print(f"Loaded {len(items)} DynaSent sentences directly from GitHub.")
items[["item_id", "sentence", "collection_round"]].head(3)

## 1. Label six sentences yourself

Read these sentences before looking at the DynaSent judgments. For each one,
choose `negative`, `neutral`, or `positive`, and give a confidence from 0.50 to
1.00. You are recording your own judgment—not guessing the majority vote.


In [ ]:
# Choose two examples at each eventual agreement level, but hide the votes.
agreement_level = items[VOTE_COLS].max(axis=1)
label_items = (
    pd.concat([
        items.loc[agreement_level.eq(level)].sample(2, random_state=2027 + level)
        for level in [3, 4, 5]
    ])
    .sample(frac=1, random_state=2027)
    .reset_index(drop=True)
)

label_items[["item_id", "sentence"]]

In [ ]:
# Fill in six labels and six confidence values before continuing.
# Example: YOUR_LABELS = ["neutral", "positive", ...]
YOUR_LABELS = ["", "", "", "", "", ""]
YOUR_CONFIDENCE = [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]

my_judgments = label_items[["item_id", "sentence"]].copy()
my_judgments["my_label"] = YOUR_LABELS
my_judgments["my_confidence"] = YOUR_CONFIDENCE
my_judgments

### Now reveal the five judgments


In [ ]:
revealed = my_judgments.merge(
    items[["item_id", *VOTE_COLS]], on="item_id", how="left"
)
revealed

**Choose one sentence for which the five judgments capture something your single label does not.**

*Your response:*  


## 2. Build the item-level table

For each sentence we want the observed vote proportions, the majority label,
and an agreement pattern. With five judgments, the possible patterns are `5–0`,
`4–1`, and `3–2`.

Those are the only possibilities **because this teaching subset is restricted
to sentences receiving votes in no more than two sentiment categories**. A
split such as `3–1–1` cannot occur here. This is a selection rule for our subset,
not a general property of multiclass labels.

The class proportions are empirical measurements from these five judgments.
They are not being declared the population's true sentiment probabilities.


In [ ]:
summary = items.copy()

# The 5–0 / 4–1 / 3–2 notation assumes at most two categories receive votes.
# Verify that deliberate subset restriction before constructing the pattern.
n_categories_used = summary[VOTE_COLS].gt(0).sum(axis=1)
assert n_categories_used.le(2).all()

# Turn vote counts into observed proportions.
for label in LABELS:
    summary[f"q_{label}"] = summary[f"{label}_votes"] / 5

# The column with the largest count determines the majority label.
summary["majority_label"] = (
    summary[VOTE_COLS].idxmax(axis=1).str.removesuffix("_votes")
)

# Sort the three counts; the two largest define 5–0, 4–1, or 3–2.
ordered_counts = np.sort(summary[VOTE_COLS].to_numpy(), axis=1)[:, ::-1]
summary["agreement_pattern"] = [
    f"{largest}–{second}" for largest, second, _ in ordered_counts
]

Q_COLS = [f"q_{label}" for label in LABELS]
assert np.allclose(summary[Q_COLS].sum(axis=1), 1)
assert set(summary["agreement_pattern"]) == {"5–0", "4–1", "3–2"}

summary[["sentence", *VOTE_COLS, *Q_COLS, "majority_label", "agreement_pattern"]].head()

**What information is lost when five judgments become one majority label?**

*Your response:*  


## 3. Compare random and high-disagreement sentences

The selection rules are explicit:

- **random:** three rows sampled with `random_state=2027`;
- **high disagreement:** three rows sampled from the `3–2` items with the same
  random state.

Run the supplied code, then change `DISPLAY_COLS` if another column would help
you interpret the examples.


In [ ]:
DISPLAY_COLS = ["sentence", *VOTE_COLS, "agreement_pattern"]

random_examples = summary.sample(3, random_state=2027)
high_disagreement_examples = (
    summary.query("agreement_pattern == '3–2'")
    .sample(3, random_state=2027)
)

print("RANDOM EXAMPLES")
display(random_examples[DISPLAY_COLS])
print("HIGH-DISAGREEMENT EXAMPLES (3–2)")
display(high_disagreement_examples[DISPLAY_COLS])

**Name one recurring source of ambiguity, if one appears. Why should we avoid claiming that it explains every 3–2 item?**

*Your response:*  


## 4. Compare the two construction rounds

DynaSent's rounds were built differently. Make both counts and within-round
percentages visible. The denominator for each percentage row is the number of
items from that construction round in this teaching subset.


In [ ]:
ROW_VARIABLE = "collection_round"
COLUMN_VARIABLE = "agreement_pattern"

round_counts = pd.crosstab(
    summary[ROW_VARIABLE], summary[COLUMN_VARIABLE]
)
round_percent = (
    pd.crosstab(
        summary[ROW_VARIABLE], summary[COLUMN_VARIABLE], normalize="index"
    )
    .mul(100)
    .round(1)
)

print("COUNTS")
display(round_counts)
print("WITHIN-ROUND PERCENTAGES")
display(round_percent)

**Write one descriptive comparison. Why is a causal sentence about round inappropriate?**

*Your response:*  


## 5. Score one forecast against two targets

Each row contains a frozen model probability vector

$$p=(p_{\mathrm{neg}},p_{\mathrm{neutral}},p_{\mathrm{positive}}).$$

We hold that forecast fixed and change the target:

- **hard target:** the one-hot encoding of the majority label;
- **soft target:** the three observed vote proportions.

For either target vector $t$, the unnormalized three-class Brier score is

$$B(p,t)=\sum_{k=1}^3(p_k-t_k)^2.$$

Scikit-learn's `label_binarize` creates the hard target. The standard
classification Brier scorer expects one observed class per row, so we apply the
four-line vector formula ourselves to support the empirical soft targets too.


In [ ]:
model_probability = summary[MODEL_COLS].to_numpy()

# Hard target: one 1 and two 0s, based on the majority label.
hard_target = label_binarize(summary["majority_label"], classes=LABELS)

# Soft target: the observed proportions among five judgments.
soft_target = summary[Q_COLS].to_numpy()

def multiclass_brier(probability, target):
    """One unnormalized three-class Brier score per row."""
    squared_difference = np.square(probability - target)
    return squared_difference.sum(axis=1)

summary["hard_brier"] = multiclass_brier(model_probability, hard_target)
summary["soft_brier"] = multiclass_brier(model_probability, soft_target)

# Package cross-check: scikit-learn accepts the observed majority class here.
hard_mean_sklearn = brier_score_loss(
    summary["majority_label"], model_probability,
    labels=LABELS, scale_by_half=False
)
assert np.isclose(hard_mean_sklearn, summary["hard_brier"].mean())

assert summary[["hard_brier", "soft_brier"]].ge(0).all().all()
assert summary[["hard_brier", "soft_brier"]].le(2).all().all()

summary[[
    "sentence", "majority_label", *Q_COLS, *MODEL_COLS,
    "hard_brier", "soft_brier"
]].head()

In [ ]:
# Inspect a sentence for which target construction matters substantially.
summary["score_difference"] = (
    summary["hard_brier"] - summary["soft_brier"]
).abs()

case = summary.nlargest(1, "score_difference")
case[[
    "sentence", *VOTE_COLS, *MODEL_COLS,
    "hard_brier", "soft_brier", "score_difference"
]]

**Why do the two scores differ? Does a lower soft-target score prove that the soft target is always correct?**

*Your response:*  


## 6. What assumptions would justify averaging raters?

Discuss independence and exchangeability separately. Several labels are not
automatically several independent, interchangeable replications.


| Design | Is independence plausible? What evidence is needed? | Is exchangeability plausible? Why? |
|---|---|---|
| Five crowd raters use the same prompt and cannot see one another's answers |  |  |
| One expert and four novices label separately |  |  |
| A student and two frozen classifiers forecast the same sentences |  |  |


**Even if an iid rater model is not justified, what do the observed vote proportions still describe?**

*Your response:*  


## Exit ticket


**In one or two sentences, state what the empirical vote proportions measure and name one choice made when those measurements become a benchmark target.**

*Your response:*  


## Before leaving

- [ ] Your six judgments were recorded before the votes were revealed.
- [ ] The item-level checks pass.
- [ ] You inspected actual random and 3–2 sentences.
- [ ] Your round comparison shows denominators and avoids a causal claim.
- [ ] You can point to the exact code that creates both Brier-score columns.

**HW2 begins here:** it formalizes the distinction among an individual label,
an empirical vote distribution, a majority target, and a forecast.
